## SKILLS EXTRACTION

## TEXT PREPROCESSING

In [53]:
import pandas as pd
import string

#load the json file that contains the scrapped offers as pandas df
df = pd.read_json('jobs.json')
#drop the job_title column 
df.drop("job_title", axis=1,inplace= True)
df.head()
df.dropna()

,job_description
0,Job Title: Python Backend EngineerEmployment T...
1,Requisition ID: 251759Join a purpose driven wi...
2,Requisition ID: 251792Join a purpose driven wi...
3,Seeking a Python Developer with 5+ years of ex...
4,Experience Required: 8–10 YearsRole OverviewWe...
5,"Java, Python, AWS app onboarding and Postgres ..."
6,Python Data Engineer/DeveloperToronto - Hybrid...
7,About AscendionAscendion is a full-service dig...
8,Job Title: Backend Python Engineer Location: T...
9,Job Title: Python Developer – Banking SectorLo...


##CONVERT TO LOWER CASE

In [54]:
df["cleaned_text"] = df["job_description"].str.lower()

df.head()

,job_description,cleaned_text
0,Job Title: Python Backend EngineerEmployment T...,job title: python backend engineeremployment t...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id: 251759join a purpose driven wi...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id: 251792join a purpose driven wi...
3,Seeking a Python Developer with 5+ years of ex...,seeking a python developer with 5+ years of ex...
4,Experience Required: 8–10 YearsRole OverviewWe...,experience required: 8–10 yearsrole overviewwe...


## REMOVE ALL POUNCTIATION

In [55]:
punctuations = string.punctuation
punctuations

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [56]:
def remove_ponc(text):
    punctuations = string.punctuation
    #punctuations += "–" # to handle that character beacuse it doesnt exist in the punctuation
    #translate method applies a translation table on a string (str to replace , what to replace with,str to delete)
    #and skipps NAN values
    if text:
         text = text.translate(str.maketrans('','',punctuations))
    return text
    
#apply it on the cleaned text
df['cleaned_text'] = df['cleaned_text'].apply(lambda x : remove_ponc(x))
df.head()

,job_description,cleaned_text
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id 251759join a purpose driven win...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join a purpose driven win...
3,Seeking a Python Developer with 5+ years of ex...,seeking a python developer with 5 years of exp...
4,Experience Required: 8–10 YearsRole OverviewWe...,experience required 8–10 yearsrole overviewwe ...


## REMOVE STOPWORDS

**Stop words are common words in any language that occur with a high frequency but carry much less substantive information about the meaning of a phrase.**

**Examples of some common stop words include:**
**a, the, and , or , of , on , this , we , were, is, not …**

In [57]:
import nltk
nltk.download('stopwords', quiet=True)

True

In [58]:
from nltk.corpus import stopwords

stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [59]:
stop_words = set(stopwords.words('english'))
def remove_stop_words(text):
    if text :
        text = " ".join([word for word in text.split() if word not in stop_words])
    return text
df["cleaned_text"] = df["cleaned_text"].apply(lambda text : remove_stop_words(text))
df.head()

,job_description,cleaned_text
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id 251759join purpose driven winni...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join purpose driven winni...
3,Seeking a Python Developer with 5+ years of ex...,seeking python developer 5 years experience ba...
4,Experience Required: 8–10 YearsRole OverviewWe...,experience required 8–10 yearsrole overviewwe ...


## REMOVE FREQUENT WORDS

**Frequency words are words that occur very often in your specific dataset, not necessarily stopwords.**

They can include stopwords (like “the”) but also domain-specific common words.

These are found by counting occurrences of each word in your dataset.

In [60]:
from collections import Counter
#we create like an empty dictionnary counter 
word_count = Counter()

for text in df["cleaned_text"]:
    if text:
        for word in text.split():
          word_count[word] += 1
word_count.most_common(10)


[('experience', 147),
 ('data', 98),
 ('work', 93),
 ('team', 76),
 ('software', 76),
 ('python', 69),
 ('services', 66),
 ('development', 61),
 ('engineering', 56),
 ('skills', 56)]

In [61]:
frequent_words = set(word for (word,word_count) in word_count.most_common(1))
def remove_frequent_words(text):
    if text :
        text = " ".join([word for word in text.split() if word not in frequent_words])
    return text
df["cleaned_text"] = df["cleaned_text"].apply(lambda text : remove_frequent_words(text)) 
df.head()

,job_description,cleaned_text
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id 251759join purpose driven winni...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join purpose driven winni...
3,Seeking a Python Developer with 5+ years of ex...,seeking python developer 5 years bankingfinanc...
4,Experience Required: 8–10 YearsRole OverviewWe...,required 8–10 yearsrole overviewwe looking dat...


## REMOVAL OF RARE WORDS

In [62]:
less_frequent = set(word for (word,word_count) in word_count.most_common()[:-20:-1]) #list[start : stop : step]
less_frequent

{'2008',
 'art',
 'artists',
 'billions',
 'canspotify',
 'creators',
 'fans',
 'forever',
 'launched',
 'listening',
 'million',
 'podcasting',
 'reasonable',
 'stage',
 'subscription',
 'today',
 'transformed',
 'unlock',
 'ways'}

In [63]:
def remove_less_frequent(text):
    if text:
        text = " ".join([word for word in text.split() if word not in less_frequent])
    return text
df["cleaned_text"] = df["cleaned_text"].apply(lambda text : remove_less_frequent(text)) 
#df["job_description"][9]

## REMOVAL OF SPECIAL CHARACTERS

In [64]:
import re
def remove_special_char(text):
    if text :
        #anything that is not a number subtitute it with " "
        text = re.sub('[^a-zA-Z0-9]'," ",text)
        #replace additional spaces with one space
        text = re.sub('\s+'," ",text)
    return text
df["cleaned_text"] = df["cleaned_text"].apply(lambda text : remove_special_char(text)) 
df.head()

,job_description,cleaned_text
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id 251759join purpose driven winni...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join purpose driven winni...
3,Seeking a Python Developer with 5+ years of ex...,seeking python developer 5 years bankingfinanc...
4,Experience Required: 8–10 YearsRole OverviewWe...,required 8 10 yearsrole overviewwe looking dat...


## STEMMING

In [65]:
from nltk.stem.porter import PorterStemmer

porter_stemmer = PorterStemmer()
def stem_words(text):
    if text:
        text = " ".join(porter_stemmer.stem(word) for word in text.split())
    return text

df["stemmed_text"] = df["cleaned_text"].apply(lambda text : stem_words(text))
df.head()

,job_description,cleaned_text,stemmed_text
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...,job titl python backend engineeremploy type fu...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id 251759join purpose driven winni...,requisit id 251759join purpos driven win team ...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join purpose driven winni...,requisit id 251792join purpos driven win team ...
3,Seeking a Python Developer with 5+ years of ex...,seeking python developer 5 years bankingfinanc...,seek python develop 5 year bankingfinanci serv...
4,Experience Required: 8–10 YearsRole OverviewWe...,required 8 10 yearsrole overviewwe looking dat...,requir 8 10 yearsrol overvieww look data engin...


## LEMMATIZATION ans POS TAGGING

**LEMMATIZATION IS MORE ACCURATE THAN STEMMING BECAUSE STEMMING CAN GIVE ROOTS THAT DOES NOT EXIST IN REALITY**

**Parts of Speech (PoS) tagging is a fundamental task in Natural Language Processing (NLP) where each word in a sentence is assigned a grammatical category such as noun, verb, adjective or adverb.**

In [66]:
import nltk
# For WordNet lemmatizer
nltk.download('wordnet')

# Optional: for tokenization (safer than split)
nltk.download('punkt')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\wiame\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wiame\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [67]:
# For POS tagging
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\wiame\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [68]:
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

In [69]:
lemmatizer = WordNetLemmatizer()
word_net_map = {
    "N": wordnet.NOUN,
    "V": wordnet.VERB,
    "J": wordnet.ADJ,
    "R": wordnet.ADV
}

def lemmatize_text(text):
    #find POS tag (part of speech)
    #It tells you the role of each word (noun, verb, adjective, etc.).
    if text :
       pos_text = pos_tag(text.split())
       text = " ".join([
           lemmatizer.lemmatize(word,word_net_map.get(pos[0],wordnet.NOUN))
           for word,pos in pos_text
       ])
    return text
        ## we added wordnet.NOUN in case we didnt find the category we take it as noun 

In [70]:
df["cleaned_text"] = df["cleaned_text"].apply(lambda text : lemmatize_text(text))
df.head()

,job_description,cleaned_text,stemmed_text
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...,job titl python backend engineeremploy type fu...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id 251759join purpose drive win te...,requisit id 251759join purpos driven win team ...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join purpose drive win te...,requisit id 251792join purpos driven win team ...
3,Seeking a Python Developer with 5+ years of ex...,seek python developer 5 year bankingfinancial ...,seek python develop 5 year bankingfinanci serv...
4,Experience Required: 8–10 YearsRole OverviewWe...,require 8 10 yearsrole overviewwe look data en...,requir 8 10 yearsrol overvieww look data engin...


In [71]:
df.sample(frac=1).head(10)

,job_description,cleaned_text,stemmed_text
21,Full Stack Python DeveloperJob Type: ContractC...,full stack python developerjob type contractco...,full stack python developerjob type contractco...
13,Key Responsibilities Design and develop softwa...,key responsibility design develop software sol...,key respons design develop softwar solut use p...
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...,job titl python backend engineeremploy type fu...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join purpose drive win te...,requisit id 251792join purpos driven win team ...
14,5+ years of experience with Python Development...,5 year python developmentexperience create lin...,5 year python developmentexperi creat line bra...
36,Requisition ID: 250903Join a purpose driven wi...,requisition id 250903join purpose drive win te...,requisit id 250903join purpos driven win team ...
8,Job Title: Backend Python Engineer Location: T...,job title backend python engineer location tor...,job titl backend python engin locat toronto on...
19,Location Address: Hybrid – Toronto – 2 days/we...,location address hybrid toronto 2 daysweek mon...,locat address hybrid toronto 2 daysweek monday...
22,Canonical is a leading provider of open source...,canonical lead provider open source software o...,canon lead provid open sourc softwar oper syst...
27,Job Title: Sr Python DeveloperLocation : Toron...,job title sr python developerlocation toronto ...,job titl sr python developerloc toronto 5 day ...


## REMOVAL OF URLS

In [72]:
txt = "https://www.blogss.net is a url for the blog website"

def remove_url(text):
    if text :
       return re.sub(r'https?://\S+|www\.\S+', '', text)

In [73]:
remove_url(txt)

' is a url for the blog website'

In [74]:
df["cleaned_text"] = df["cleaned_text"].apply(lambda text : remove_url(text))

In [75]:
df.head()

,job_description,cleaned_text,stemmed_text
0,Job Title: Python Backend EngineerEmployment T...,job title python backend engineeremployment ty...,job titl python backend engineeremploy type fu...
1,Requisition ID: 251759Join a purpose driven wi...,requisition id 251759join purpose drive win te...,requisit id 251759join purpos driven win team ...
2,Requisition ID: 251792Join a purpose driven wi...,requisition id 251792join purpose drive win te...,requisit id 251792join purpos driven win team ...
3,Seeking a Python Developer with 5+ years of ex...,seek python developer 5 year bankingfinancial ...,seek python develop 5 year bankingfinanci serv...
4,Experience Required: 8–10 YearsRole OverviewWe...,require 8 10 yearsrole overviewwe look data en...,requir 8 10 yearsrol overvieww look data engin...


In [78]:
#saving the data frame after text preprocessing into csv file
import pandas as pd
df["cleaned_text"].to_csv("my_cleaned_text.csv",index= False)